In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != 'QB-PIER':
    PROJECT_ROOT = Path('../../../').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from analytics.attacks.mechanistic_visualizations import (
    OUTPUT_DIR, VALIDATION_KEYS, model_specs, validate_checkpoints, build_cohorts,
    trace_self_layerwise, trace_interfaces, trace_segmentation,
    trace_source_directions, trace_residual_branches, save_core_tables,
    generate_figure_package,
)

MODEL_ROOT = PROJECT_ROOT / 'logs' / 'ByModel'
OUTPUT_PATH = OUTPUT_DIR
specs = model_specs()

In [ ]:
# Crawl only: no model is loaded in this block.
model_dirs = sorted(path.parent for path in (PROJECT_ROOT / 'logs').rglob('.model'))
selected_specs = {key: specs[key] for key in VALIDATION_KEYS}
print(f'{len(model_dirs)} model directories discovered through .model anchors')
[(key, spec.model_dir, spec.raw_attack_name) for key, spec in selected_specs.items()]

In [ ]:
# Expensive preparation: checkpoint validation, paired cohorts and traces.
alignment, architecture = validate_checkpoints(specs, keys=tuple(selected_specs))
cohorts, cohort_summary = build_cohorts(specs)
layerwise = trace_self_layerwise(cohorts, specs)
interface = trace_interfaces(cohorts, specs)
segmentation = trace_segmentation(cohorts, specs)
directions = trace_source_directions(cohorts, specs)
residual, residual_scales = trace_residual_branches(cohorts, specs)
core_paths = save_core_tables(
    alignment, architecture, cohort_summary, layerwise, interface, segmentation,
    directions, residual, residual_scales, OUTPUT_PATH,
)
core_paths

In [ ]:
# Generate every independent PDF/SVG/PNG plus CSV, method, caption and exact wrapper.
figure_manifest = generate_figure_package(
    alignment, architecture, cohort_summary, layerwise, interface, segmentation,
    directions, residual, OUTPUT_PATH,
)
figure_manifest